---
title: Aggregate a dataset to organisation units and import into DHIS2
short_title: Aggregate to org units
---

Open Climate Service ships a reusable workflow, `aggregate_to_dhis2_json`, that runs entirely **server-side**: it loads a published dataset over a time range, aggregates it within each organisation-unit polygon, and returns a ready-to-import DHIS2 `dataValueSet`.

In this notebook we call that workflow and then import its result into DHIS2 with the [dhis2-python-client](https://github.com/dhis2/dhis2-python-client) — the same client used in the [Importing data values](../guides/import-data/import-data-values.ipynb) guide.

Needs the `open-climate-service` client (see the [section intro](intro.md)) and a running instance.

In [ ]:
from open_climate_service import ClimateService

service = ClimateService("https://my-instance.example.org")

## 1) Organisation unit boundaries

The workflow aggregates to whatever GeoJSON features you provide. Each feature's `id` **must be the DHIS2 organisation unit UID**, so the result can be imported into DHIS2 without any remapping. See the [Organisation units](../guides/org-units/intro.md) guide for how to export these from DHIS2 as GeoJSON.

In [ ]:
import json
from pathlib import Path

# A GeoJSON FeatureCollection of your org units, each feature "id" = DHIS2 org unit UID.
org_units = json.loads(Path("org-units.geojson").read_text())
print(len(org_units["features"]), "organisation units")

## 2) Run the aggregation workflow

We call the workflow with `execute()`. The key arguments are:

- `dataset_id` — a published dataset (see [Connect and explore](connect-and-explore.ipynb)).
- `temporal_extent` — `[start, end]` dates.
- `geometries` — the org-unit GeoJSON.
- `data_element_id` — the DHIS2 data element the values belong to.
- `method` — the spatial statistic: `mean` (default), `min`, `max`, or `sum`.
- `period_type` — how each time step is turned into a DHIS2 period (`month`, `week`, `day`, …).

For a JSON result like this one, `execute()` returns the parsed payload as a `dict`.

In [ ]:
result = service.execute(
    {
        "agg": {
            "process_id": "aggregate_to_dhis2_json",
            "arguments": {
                "dataset_id": "era5land_temperature_monthly",   # a published dataset id
                "temporal_extent": ["2025-01-01", "2025-12-31"],
                "geometries": org_units,
                "data_element_id": "BXgDHhPdFVU",   # DHIS2 data element to import into
                "method": "mean",                     # mean (default), min, max, or sum
                "period_type": "month",
            },
            "result": True,
        }
    }
)
result["dataValues"][:3]

The workflow has already filled in `orgUnit`, `period`, `value`, and `dataElement` for every cell — the payload is a valid DHIS2 `dataValueSet`, ready to import as-is.

## 3) Import into DHIS2

Connect to DHIS2 and post the payload. (Replace the demo server and credentials with your own.)

In [ ]:
from dhis2_client import DHIS2Client
from dhis2_client.settings import ClientSettings

client = DHIS2Client(
    settings=ClientSettings(
        base_url="https://climate.im.dhis2.org/climate-tools-42",
        username="admin",
        password="district",
    )
)

res = client.post_data_value_set(result)
res["response"]["importCount"]

The data element and organisation units referenced in the payload must already exist in DHIS2. See [Importing data values](../guides/import-data/import-data-values.ipynb) for preparing metadata, dry runs, and troubleshooting import conflicts.

## Next steps

- [Prepare data for Chap](prepare-data-for-chap.ipynb) — get the same aggregation as a Chap-ready CSV instead.